In [1]:
!pip install pyngrok

In [2]:
dataset_path = "/kaggle/input/datasets/lyashenkovilya/cats-latent-flw-matching/cats_latents_16x16x8.npy"

# Decoder

In [3]:
cp /kaggle/input/models/lyashenkovilya/strongson/keras/default/1/models.py ./

In [4]:
import json

with open("/kaggle/input/models/lyashenkovilya/strongson/keras/default/1/config.json", 'r') as j:
    config = json.load(j)

In [5]:
from models import get_vae_from_config
vae = get_vae_from_config(config)
vae.load_weights("/kaggle/input/models/lyashenkovilya/strongson/keras/default/1/StrongSon.weights.h5")
decoder = vae.decoder

2026-04-12 10:56:49.128838: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775991409.311008      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775991409.362613      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775991409.788890      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775991409.788939      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775991409.788941      23 computation_placer.cc:177] computation placer alr

# Unet

In [6]:
import numpy as np
import tensorflow as tf
import keras

In [7]:
from tensorflow.keras import layers
from keras import ops # Универсальные операции Keras 3

def time_embedding(t, dim=128):
    # Вместо tf.expand_dims используем Reshape или keras.ops
    # Нам нужно превратить (batch, 1) в (batch, 1, 1) для Dense
    x = layers.Reshape((1,))(t) 
    x = layers.Dense(dim, activation='swish')(x)
    x = layers.Dense(dim)(x)
    return x

def residual_block(x, t_emb, filters):
    shortcut = layers.Conv2D(filters, kernel_size=1, padding='same')(x)
    
    # Впрыск времени: t_emb имеет форму (batch, dim)
    # Нам нужно раздуть его до (batch, 1, 1, filters), чтобы прибавить к свертке
    t = layers.Dense(filters)(t_emb)
    t = layers.Reshape((1, 1, filters))(t)
    
    x = layers.Conv2D(filters, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    
    # Теперь формы x (h, w, f) и t (1, 1, f) совместимы для сложения
    x = layers.Add()([x, t]) 
    
    x = layers.Conv2D(filters, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    return layers.Activation('swish')(x)

def build_unet_fm(input_shape=(16, 16, 8)):
    img_input = layers.Input(shape=input_shape, name = 'x')
    time_input = layers.Input(shape=(1,), name = 't')
    
    # 1. Embed time
    t_emb = time_embedding(time_input)
    
    # 2. Downsample
    x1 = residual_block(img_input, t_emb, 64)
    x2 = layers.AveragePooling2D((2, 2))(x1)
    x2 = residual_block(x2, t_emb, 128)
    
    # 3. Bottleneck
    b = residual_block(x2, t_emb, 256)
    
    # 4. Upsample
    u1 = layers.UpSampling2D((2, 2))(b)
    u1 = layers.Concatenate()([u1, x1]) # Skip connection
    u1 = residual_block(u1, t_emb, 64)
    
    # 5. Output (Скорость v)
    # Важно: выход должен иметь ту же размерность, что и вход
    output = layers.Conv2D(input_shape[-1], kernel_size=3, padding='same')(u1)
    
    return tf.keras.Model({"x":img_input, "t":time_input}, output)

In [8]:
unet = build_unet_fm()
optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3, # Давай начнем с агрессивного, чтобы "размазать" гистограмму
    weight_decay=1e-4   # Это заставит веса быть более структурными
)

unet.compile(optimizer=optimizer, loss='mse')

# Training

## Dataset

In [9]:
def map_func(*args):
    x = args[0]
    time = tf.random.uniform([], 0, 1)
    noise = tf.random.normal(tf.shape(x))
    point = time*x + (1 - time)*noise
    u = x - noise
    return {"x":point, "t":time}, u

def get_latent_dataset(path, batch_size=64):
    # Загружаем массив из файла
    data = np.load(path).astype('float32')
    mean = np.mean(data)
    std= np.std(data)
    
    # Создаем TF датасет прямо из массива в памяти
    ds = tf.data.Dataset.from_tensor_slices((data - mean)/std)
    
    # Шафл, батчи и prefetch (чтобы GPU не простаивал)
    ds = ds.shuffle(buffer_size=len(data))
    ds = ds.map(map_func)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds, mean, std


diffusion_dataset, mean, std = get_latent_dataset(dataset_path, 64)

## TensorBoard

In [10]:
import datetime
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = "/kaggle/working/log/" + current_time + "/"
check_points = log_dir + "/checkpoints/"

In [11]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ngrok_api_key = user_secrets.get_secret("ngrok-api-key")

In [12]:
from pyngrok import ngrok
ngrok.set_auth_token(ngrok_api_key)

In [13]:
%load_ext tensorboard

In [14]:
import os
os.system(f"tensorboard --logdir {log_dir} --port 6006 &")
public_url = ngrok.connect(6006)
print(public_url)

NgrokTunnel: "https://goggles-wildfowl-silica.ngrok-free.dev" -> "http://localhost:6006"


/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:292: SyntaxWarning: invalid escape sequence '\s'
  "[`\000-\040\177-\240\s]+",
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:339: SyntaxWarning: invalid escape sequence '\s'
  style = re.compile('url\s*\(\s*[^\s)]+?\s*\)\s*').sub(' ', style)
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:354: SyntaxWarning: invalid escape sequence '\s'
  if not re.match("^\s*([-\w]+\s*:[^:;]*(;\s*|$))*$", style):
/usr/local/lib/python3.12/dist-packages/tensorboard/_vendor/bleach/sanitizer.py:358: SyntaxWarning: invalid escape sequence '\w'
  for prop, value in re.findall('([-\w]+)\s*:\s*([^:;]*)', style):


In [15]:
def sample(model, noise, num_steps=50):
    dt = 1.0 / num_steps
    x = noise
    for i in range(num_steps):
        t = tf.cast(i / num_steps, tf.float32)
        t_input = tf.fill((noise.shape[0], 1), t)
        v = model({"x": x, "t": t_input}, training=False)
        x = x + v * dt

    return x

In [16]:
class VisualizerCallback(keras.callbacks.Callback):
    def __init__(self, log_dir, decoder, mean, std, every = 10, n = 3):
        super().__init__()
        self.decoder = decoder
        self.file_writer = tf.summary.create_file_writer(log_dir + "/images")
        self.every = every
        self.n = n
        self.mean, self.std = mean, std

    def on_epoch_end(self, epoch, logs=None):
        latent_shape = self.decoder.input_shape[1:]
        if epoch % self.every == 0:
            noise = tf.random.normal(shape=(self.n, *latent_shape))
            x = sample(self.model, noise, num_steps = 100)*self.std + self.mean
            samples = (self.decoder(x) + 1.0)/2.0

            with self.file_writer.as_default():
                tf.summary.image("Samples", samples, step=epoch)

In [17]:
from keras.callbacks import TensorBoard, EarlyStopping, ModelCheckpoint

tensorboard_callback = TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,
        write_graph=True,
    )

early_stop = EarlyStopping(
    monitor='loss',
    patience=400,
    verbose=1,
    mode='min',
    restore_best_weights=True
)

checkpoint_callback = ModelCheckpoint(
    filepath=check_points + '_{epoch:02d}_{loss:.4f}.weights.h5',
    save_weights_only=True,
    monitor='loss',
    mode='min',
    save_best_only=True
)
v_callback = VisualizerCallback(log_dir, decoder, mean, std)

## Training

In [18]:
unet.fit(diffusion_dataset, epochs=1200, callbacks = [checkpoint_callback, tensorboard_callback, early_stop, v_callback])

Epoch 1/1200


2026-04-12 10:57:20.220899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775991440.244361     103 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775991440.252350     103 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775991440.270502     103 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775991440.270547     103 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775991440.270553     103 computation_placer.cc:177] computation placer alr

157/157 ━━━━━━━━━━━━━━━━━━━━ 30s 92ms/step - loss: 1.6706
Epoch 2/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.2119
Epoch 3/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.1520
Epoch 4/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.1279
Epoch 5/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.1051
Epoch 6/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0943
Epoch 7/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0823
Epoch 8/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0696
Epoch 9/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0551
Epoch 10/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0469
Epoch 11/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - loss: 1.0386
Epoch 12/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0341
Epoch 13/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0326
Epoch 14/1200
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 1.0302
Epoch 15/1200
157/157 ━━━━━

In [19]:
unet.save(f"/kaggle/working/best_{current_time}_.keras")